# Chatbot - sentence embedding módszerrel
szöveg → embedding (Sentence-BERT) → cosine

In [ ]:
# Sentence-BERT és szükséges csomagok
!pip install -q sentence-transformers

In [ ]:
# törli a korábbi fájlokat
import os
import re

# Mappa (Colab alapértelmezett)
folder_path = "/content"

# Regex: .xlsx, .xlsx.1, .xlsx.2, stb.
pattern = re.compile(r".*\.xlsx(\.\d+)?$")

deleted = []

for filename in os.listdir(folder_path):
    if pattern.match(filename):
        file_path = os.path.join(folder_path, filename)
        try:
            os.remove(file_path)
            deleted.append(filename)
        except Exception as e:
            print(f"Hiba: {filename} -> {e}")

print("Törölt fájlok:")
for f in deleted:
    print(f)

Törölt fájlok:
sportok_leiras.xlsx


In [ ]:
# Excel letöltése
!wget https://raw.githubusercontent.com/minorharpman/ai_prog_pub/main/hirlevel_11/sportok_leiras.xlsx

import pandas as pd

# Excel beolvasása
df = pd.read_excel("sportok_leiras.xlsx")

# Cím + Szöveg összefűzése (ez lesz a tudásbázis)
df["adat"] = df["Sport"].fillna("") + " " + df["Description"].fillna("")

df["adat"].head()

--2026-03-18 13:09:42--  https://raw.githubusercontent.com/minorharpman/ai_prog_pub/main/hirlevel_11/sportok_leiras.xlsx
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 13848 (14K) [application/octet-stream]
Saving to: ‘sportok_leiras.xlsx.1’

sportok_leiras.xlsx 100%[===================>]  13.52K  --.-KB/s    in 0.002s  

2026-03-18 13:09:42 (5.41 MB/s) - ‘sportok_leiras.xlsx.1’ saved [13848/13848]



,adat
0,Pingpong (Asztalitenisz) Gyors ütős sport kis ...
1,Tenisz (fedett pályás) Hálóval elválasztott pá...
2,"Tollaslabda Ütős sport, ahol tolllabdát ütnek ..."
3,Squash (Fallabda) Zárt pályán játszott ütős sp...
4,Padel Falakkal körülvett pályán játszott ütős ...


In [ ]:
#teszt
talalat = df[df["Sport"] == "Íjászat (teremben)"]

# Description kiírása
if not talalat.empty:
    print(talalat.iloc[0]["Description"])
else:
    print("Nincs 'Íjászat' sport a táblában.")

Íjjal történik,  nyilak,  célzás, koncentráció


#Embedding modell betöltése

In [ ]:
from sentence_transformers import SentenceTransformer

# Multilingual modell (magyarhoz is jó!)
model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

print("Modell betöltve")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Modell betöltve


#Dokumentum embeddingek előállítása

In [ ]:
# A teljes tudásbázis vektorizálása
# Ez egyszer történik meg → később gyors lesz a keresés

dokumentumok = df["adat"].tolist()

# Embeddingek generálása
dokumentum_embeddingek = model.encode(dokumentumok, show_progress_bar=True)

print("Embeddingek elkészültek:", len(dokumentum_embeddingek))

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Embeddingek elkészültek: 100


#Koszinusz hasonlóság függvény

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def legjobb_talalat(kerdes_embedding, dokumentum_embeddingek):
    """
    Kiszámolja a kérdés és az összes dokumentum közti hasonlóságot,
    majd visszaadja a legjobb találat indexét és értékét.
    """

    hasonlosagok = cosine_similarity([kerdes_embedding], dokumentum_embeddingek)[0]

    legjobb_index = hasonlosagok.argmax()
    legjobb_ertek = hasonlosagok[legjobb_index]

    return legjobb_index, legjobb_ertek

In [ ]:
import numpy as np

def top_k_talalatok(kerdes_embedding, dokumentum_embeddingek, k=3):
    """
    Visszaadja a top-k legjobb találat indexeit és pontszámait.

    Paraméterek:
    - kerdes_embedding: a kérdés vektora
    - dokumentum_embeddingek: összes dokumentum vektora
    - k: hány találatot adjunk vissza

    Visszatérés:
    - lista (index, hasonlóság) párokkal
    """

    # Koszinusz hasonlóság számítása
    hasonlosagok = cosine_similarity([kerdes_embedding], dokumentum_embeddingek)[0]

    # Top-k index kiválasztása (legnagyobb értékek)
    top_k_indexek = np.argsort(hasonlosagok)[-k:][::-1]

    # (index, pontszám) párok
    eredmenyek = [(i, hasonlosagok[i]) for i in top_k_indexek]

    return eredmenyek

#Chatbot függvény

In [ ]:
def chatbot_valaszok(kerdes, k=2, kuszob=0.3):
    """
    Több találatos chatbot:
    - visszaadja a top-k válaszokat

    Paraméterek:
    - kerdes: felhasználó kérdése
    - k: hány találatot adjunk vissza
    - kuszob: minimum hasonlóság
    """

    # 1. Kérdés embedding
    kerdes_embedding = model.encode(kerdes)

    # 2. Top-k találatok
    talalatok = top_k_talalatok(kerdes_embedding, dokumentum_embeddingek, k)

    valaszok = []

    print(f"\nKérdés: {kerdes}")
    print("Legjobb találatok:\n")

    # 3. Találatok feldolgozása
    for i, (index, pontszam) in enumerate(talalatok, 1):

        print(f"{i}. találat | Hasonlóság: {pontszam:.3f}")

        if pontszam >= kuszob:
            szoveg = df.iloc[index]["adat"]
            print(szoveg[:300], "...")  # csak első 300 karakter
            valaszok.append(szoveg)
        else:
            print("Túl alacsony hasonlóság")

        print("-" * 50)

    if not valaszok:
        return ["Sajnos nem találtam megfelelő választ."]

    return valaszok

 # Chatbot teszt
## példák:
* milyen relaxációs tevékenységek vannak?
* milyen technikai sportok vannak?
* milyen erősítő sportok vannak?
* milyen társasági sportok vannak?


In [ ]:
kerdes = input("Én: ")
print("___________")

valasz = chatbot_valaszok(kerdes)
#print("Bot:", valasz)

Én: milyen erősítő sportok vannak?
___________

Kérdés: milyen erősítő sportok vannak?
Legjobb találatok:

1. találat | Hasonlóság: 0.660
Súlyzózás (Klasszikus testépítés) Izolált izomépítésre fókuszáló edzés súlyzókkal. Gyakorlatok külön izomcsoportokra céloznak. Progresszív terhelés jellemzi. ...
--------------------------------------------------
2. találat | Hasonlóság: 0.654
BodyArt Erősítő és nyújtó elemeket kombináló edzés. Légzés és testkontroll hangsúlyos. Funkcionális mozgásra épít. ...
--------------------------------------------------


In [ ]:
print(chatbot_valaszok("Erősítés, súlyzó"))
print()
print(chatbot_valaszok("Relaxációs tevékenység"))
print()
print(chatbot_valaszok("Csapatjáték"))


Kérdés: Erősítés, súlyzó
Legjobb találatok:

1. találat | Hasonlóság: 0.495
Súlyzózás (Klasszikus testépítés) Izolált izomépítésre fókuszáló edzés súlyzókkal. Gyakorlatok külön izomcsoportokra céloznak. Progresszív terhelés jellemzi. ...
--------------------------------------------------
2. találat | Hasonlóság: 0.440
BodyArt Erősítő és nyújtó elemeket kombináló edzés. Légzés és testkontroll hangsúlyos. Funkcionális mozgásra épít. ...
--------------------------------------------------
['Súlyzózás (Klasszikus testépítés) Izolált izomépítésre fókuszáló edzés súlyzókkal. Gyakorlatok külön izomcsoportokra céloznak. Progresszív terhelés jellemzi.', 'BodyArt Erősítő és nyújtó elemeket kombináló edzés. Légzés és testkontroll hangsúlyos. Funkcionális mozgásra épít.']


Kérdés: Relaxációs tevékenység
Legjobb találatok:

1. találat | Hasonlóság: 0.675
Hatha jóga Testhelyzetekre és légzésre épülő mozgásforma. Rugalmasságot és mentális fókuszt fejleszt. Lassú, kontrollált gyakorlatok. ...
-------

#  3D vizualizáció – sportokkal

In [ ]:
# ==============================
# 3D vizualizáció – sportok nevekkel
# ==============================

from sklearn.decomposition import PCA
import plotly.express as px

# --- 1. SPORT NEVEK (Excel első oszlop) ---
sportok = df.iloc[:, 0].dropna().astype(str).tolist()

# --- 2. EMBEDDINGEK ---
# Ha nálad így hívják:
embeddingek = dokumentum_embeddingek  # vagy: embeddingek

# Ellenőrzés (nagyon fontos!)
assert len(sportok) == len(embeddingek), "Eltérés van a sportok és embeddingek számában!"

# --- 3. PCA → 3D ---
pca = PCA(n_components=3)
embedding_3d = pca.fit_transform(embeddingek)

# --- 4. 3D ÁBRA ---
fig = px.scatter_3d(
    x=embedding_3d[:, 0],
    y=embedding_3d[:, 1],
    z=embedding_3d[:, 2],
    text=sportok,  # <-- EZ teszi rá a sport neveket!
    title="Sportok 3D embedding térben"
)

# Szebb megjelenítés
fig.update_traces(
    marker=dict(size=5),
    textposition="top center"
)

fig.show()

# ❓ Kell-e lemmatizáció sentence embeddingnél magyar szövegre?

"""
Rövid válasz:
Nem, sentence embedding (pl. Sentence-BERT) esetén NEM kell lemmatizáció,
sőt sokszor rontja a teljesítményt.

Miért?

A Sentence-BERT (BERT-alapú modell):
- subword tokenizációt használ (nem teljes szavakat)
- képes kezelni a ragozott alakokat
- a mondat teljes jelentését veszi figyelembe

Példa:
"fáj a fogam"
"fogfájásom van"

→ nagyon hasonló embeddinget kapnak

Magyar nyelvnél:
- bár sok a toldalék (pl. "fogban", "fogból", "foghoz")
- a modell ezt jól kezeli tanulásból + subword bontásból

Ezért:

NE használj lemmatizációt, ha:
- pretrained Sentence-BERT modellt használsz
- chatbotot vagy keresőt építesz
- jelentés alapú hasonlóság kell

Miért rossz a lemmatizáció itt?
- információt veszítesz (igeidő, hangsúly)
- természetellenes szöveg lesz
- romolhat a hasonlóság számítás

Mit csinálj helyette?

✔ ajánlott:
- kisbetűsítés (lowercase)
- alap tisztítás (whitespace, extra karakterek)

❌ nem ajánlott:
- lemmatizáció
- stemming

Ökölszabály:
TF-IDF → kell lemmatizáció
Sentence embedding → NEM kell
"""